# 01 - Data Preprocessing

This notebook loads the raw `.fa` FASTA files and produces a unified, model-ready CSV dataset.

**Role B**: Dataset & Preprocessing

## 1. Overview

| File | Label | Split |
|---|---|---|
| `AMPlify_AMP_train_common.fa` | 1 (AMP) | train |
| `AMPlify_non_AMP_train_balanced.fa` | 0 (non-AMP) | train |
| `AMPlify_AMP_test_common.fa` | 1 (AMP) | test |
| `AMPlify_non_AMP_test_balanced.fa` | 0 (non-AMP) | test |

**Label convention**: `AMP = 1`, `non-AMP = 0`

In [ ]:
import pandas as pd
import os

DATA_DIR = '../data/'
OUTPUT_CSV = os.path.join(DATA_DIR, 'processed_dataset.csv')

## 2. Helper — Parse FASTA

In [ ]:
def parse_fasta(filepath):
    """Return list of amino-acid sequences from a FASTA file."""
    sequences = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('>'):
                sequences.append(line)
    return sequences

# Quick sanity check
sample = parse_fasta(os.path.join(DATA_DIR, 'AMPlify_AMP_train_common.fa'))
print(f'First sequence: {sample[0]}')
print(f'Total AMP train sequences: {len(sample)}')

## 3. Load & Combine All Four Files

In [ ]:
file_config = [
    ('AMPlify_AMP_train_common.fa',       1, 'train'),
    ('AMPlify_non_AMP_train_balanced.fa', 0, 'train'),
    ('AMPlify_AMP_test_common.fa',        1, 'test'),
    ('AMPlify_non_AMP_test_balanced.fa',  0, 'test'),
]

records = []
for filename, label, split in file_config:
    seqs = parse_fasta(os.path.join(DATA_DIR, filename))
    for seq in seqs:
        records.append({'sequence': seq, 'label': label, 'set': split})
    print(f'{filename}: {len(seqs)} sequences  (label={label}, set={split})')

df = pd.DataFrame(records)
print(f'\nTotal rows: {len(df)}')

## 4. Data Exploration

In [ ]:
print('=== Shape ===')
print(df.shape)

print('\n=== Sample rows ===')
print(df.head(6).to_string(index=False))

print('\n=== Class distribution ===')
print(df.groupby(['set', 'label']).size().rename('count').reset_index())

In [ ]:
# Sequence length statistics
df['seq_len'] = df['sequence'].str.len()
print('=== Sequence length statistics by label ===')
print(df.groupby('label')['seq_len'].describe().round(1))

## 5. Validation Checks

In [ ]:
import re

# Check for duplicates
dup_count = df.duplicated(subset='sequence').sum()
print(f'Duplicate sequences: {dup_count}')

# Check for empty sequences
empty_count = (df['sequence'].str.len() == 0).sum()
print(f'Empty sequences:     {empty_count}')

# Check valid amino acid alphabet
valid_aa = re.compile(r'^[ACDEFGHIKLMNPQRSTVWYBZXU]+$', re.IGNORECASE)
invalid = df[~df['sequence'].str.match(valid_aa)]
print(f'Invalid AA sequences: {len(invalid)}')

if dup_count == 0 and empty_count == 0 and len(invalid) == 0:
    print('\nAll checks passed!')

## 6. Save Processed Dataset

In [ ]:
df_out = df[['sequence', 'label', 'set']]
df_out.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(df_out)} rows to: {OUTPUT_CSV}')
print('\nPreview:')
print(df_out.head(4).to_string(index=False))

## 7. Summary

| Split | AMP (1) | non-AMP (0) | Total |
|-------|---------|-------------|-------|
| train | 3338    | 3338        | 6676  |
| test  | 835     | 835         | 1670  |
| **Total** | **4173** | **4173** | **8346** |

The dataset is **perfectly balanced** in both splits. Output saved to `data/processed_dataset.csv`.